# Data Cleaning for NOx emissions

### Import Packages

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import statsmodels
import plotly.io as pio
pio.renderers.default = "notebook"
import os
os.getcwd()

'/Users/maxlaurent/Documents/GitHub/Data_Science_Project'

### Load Data 

In [ ]:
df = pd.read_csv("Emissions.csv",sep="\t",low_memory=False)

df.head(10)

,countryCode,country,pollutantName,sectorCode,year,value,unit,notation,parentSectorCode,sectorName
0,AT,Austria,As,1B1b,2017,NaN,t,NR,NATIONAL TOTAL,Fugitive emission from solid fuels: Solid fuel...
1,AT,Austria,As,1B1b,2018,NaN,t,NR,NATIONAL TOTAL,Fugitive emission from solid fuels: Solid fuel...
2,AT,Austria,As,1B1b,2019,NaN,t,NR,NATIONAL TOTAL,Fugitive emission from solid fuels: Solid fuel...
3,AT,Austria,As,1B1b,2020,NaN,t,NR,NATIONAL TOTAL,Fugitive emission from solid fuels: Solid fuel...
4,AT,Austria,As,1B1b,2021,NaN,t,NR,NATIONAL TOTAL,Fugitive emission from solid fuels: Solid fuel...
5,AT,Austria,As,1B1b,2022,NaN,t,NR,NATIONAL TOTAL,Fugitive emission from solid fuels: Solid fuel...
6,AT,Austria,As,1B1b,2023,NaN,t,NR,NATIONAL TOTAL,Fugitive emission from solid fuels: Solid fuel...
7,AT,Austria,As,1B1b,2024,NaN,t,NR,NATIONAL TOTAL,Fugitive emission from solid fuels: Solid fuel...
8,AT,Austria,As,1B1c,1990,NaN,t,NR,NATIONAL TOTAL,Other fugitive emissions from solid fuels
9,AT,Austria,As,1B1c,1991,NaN,t,NR,NATIONAL TOTAL,Other fugitive emissions from solid fuels


### Using only useful colums

In [4]:
df.shape
df.columns.tolist()

['countryCode',
 'country',
 'pollutantName',
 'sectorCode',
 'year',
 'value',
 'unit',
 'notation',
 'parentSectorCode',
 'sectorName']

In [18]:
useful_cols = ["country","pollutantName","year","value","unit","sectorName"]

df_bis = pd.read_csv("Emissions.csv",sep="\t",usecols=useful_cols,low_memory=False)

df_bis.head(10)

,country,pollutantName,year,value,unit,sectorName
0,Austria,As,2017,NaN,t,Fugitive emission from solid fuels: Solid fuel...
1,Austria,As,2018,NaN,t,Fugitive emission from solid fuels: Solid fuel...
2,Austria,As,2019,NaN,t,Fugitive emission from solid fuels: Solid fuel...
3,Austria,As,2020,NaN,t,Fugitive emission from solid fuels: Solid fuel...
4,Austria,As,2021,NaN,t,Fugitive emission from solid fuels: Solid fuel...
5,Austria,As,2022,NaN,t,Fugitive emission from solid fuels: Solid fuel...
6,Austria,As,2023,NaN,t,Fugitive emission from solid fuels: Solid fuel...
7,Austria,As,2024,NaN,t,Fugitive emission from solid fuels: Solid fuel...
8,Austria,As,1990,NaN,t,Other fugitive emissions from solid fuels
9,Austria,As,1991,NaN,t,Other fugitive emissions from solid fuels


### Looking only at NOx emissions

In [19]:
df_bis["pollutantName"].dropna().unique()

<StringArray>
[                        'As',                         'BC',
            'Benzo(a) Pyrene',      'Benzo(b) Fluoranthene',
      'benzo(k) Fluoranthene',                         'Cd',
                         'CO',   'Indeno (1,2,3-cd) Pyrene',
                        'NOx',                      'PM2.5',
                         'Zn',                         'Cr',
                         'Pb',                         'Se',
                        'NH3',                       'PCBs',
                         'Cu',                         'Ni',
                        'SOx', 'PCDD/PCDF (dioxins/furans)',
                      'NMVOC',                        'HCB',
                 'Total PAHs',                       'PM10',
                        'TSP',                         'Hg',
                    'Biomass',              'Gaseous Fuels',
               'Liquid Fuels',                'Other Fuels',
                'Solid Fuels']
Length: 31, dtype: str

In [20]:
df_NOx = df_bis[df_bis["pollutantName"].str.upper().eq("NOX")].copy()

df_NOx.head()

,country,pollutantName,year,value,unit,sectorName
25945,Austria,NOx,2023,NaN,kt,Biological treatment of waste - Solid waste di...
25946,Austria,NOx,2024,NaN,kt,Biological treatment of waste - Solid waste di...
25947,Austria,NOx,1987,NaN,kt,Biological treatment of waste - Composting
25948,Austria,NOx,1988,NaN,kt,Biological treatment of waste - Composting
25949,Austria,NOx,1989,NaN,kt,Biological treatment of waste - Composting


### Removing rows with no values for NOx emissions

In [21]:
df_NOx = df_NOx.dropna(subset=["value"]).copy()

df_NOx.head()

,country,pollutantName,year,value,unit,sectorName
26544,Austria,NOx,1990,0.019140,kt,Municipal waste incineration
26545,Austria,NOx,1991,0.019140,kt,Municipal waste incineration
26887,Austria,NOx,1987,0.017732,kt,Industrial waste incineration
26888,Austria,NOx,1988,0.017732,kt,Industrial waste incineration
26889,Austria,NOx,1989,0.017732,kt,Industrial waste incineration


### Considering only total emissions in countries

In [27]:
df_NOx["sectorName"].unique()

<StringArray>
[                                                       'Municipal waste incineration',
                                                       'Industrial waste incineration',
                                                         'Clinical waste incineration',
                                                                           'Cremation',
                                                               'Open burning of waste',
                                                                         'Adjustments',
   'National total for compliance assessment  (please specify all details in the IIR)',
                        'National total for the entire territory (based on fuel sold)',
                                              'Public electricity and heat production',
                                                                  'Petroleum refining',
 ...
                                                        'Distribution of oil products',
             

For total NOx emissions of the countries, we will use 'National total for the entire territory (based on fuel sold)'.

Meaning of “based on fuel sold” : It means that emissions from fuel combustion are attributed to countries when the relevant fuel was sold in it, using the fuel quantity and pollutant-specific emission factors. 

In [39]:
total_sector = "National total for the entire territory (based on fuel sold)"

df_NOx_total = df_NOx[df_NOx["sectorName"].eq(total_sector)].copy()

df_NOx_total.head()

,country,pollutantName,year,value,unit,sectorName
30441,Austria,NOx,1987,233.633980,kt,National total for the entire territory (based...
30442,Austria,NOx,1988,228.126732,kt,National total for the entire territory (based...
30443,Austria,NOx,1989,222.619483,kt,National total for the entire territory (based...
30444,Austria,NOx,1990,213.908000,kt,National total for the entire territory (based...
30445,Austria,NOx,1991,223.452000,kt,National total for the entire territory (based...


### Filtering countries

In [40]:
df_NOx_total["country"].unique()

<StringArray>
[      'Austria',       'Belgium',      'Bulgaria',        'Cyprus',
       'Czechia',       'Germany',       'Denmark',       'Estonia',
         'Spain',       'Finland',        'France',        'Greece',
       'Croatia',       'Hungary',       'Ireland',         'Italy',
     'Lithuania',    'Luxembourg',        'Latvia',         'Malta',
   'Netherlands',        'Poland',      'Portugal',       'Romania',
        'Sweden',      'Slovenia',      'Slovakia',   'Switzerland',
       'Iceland', 'Liechtenstein',        'Norway',       'Türkiye',
         'EEA32',          'EU27']
Length: 34, dtype: str

In [50]:
considered_countries=["Norway","Sweden"]

df_NOx_clean = (df_NOx_total[df_NOx_total["country"].isin(considered_countries)]
                .sort_values(["country", "year"], ascending=[True, True]).reset_index(drop=True))

df_NOx_clean

,country,pollutantName,year,value,unit,sectorName
0,Norway,NOx,1990,196.575,kt,National total for the entire territory (based...
1,Norway,NOx,1991,190.218,kt,National total for the entire territory (based...
2,Norway,NOx,1992,194.409,kt,National total for the entire territory (based...
3,Norway,NOx,1993,199.772,kt,National total for the entire territory (based...
4,Norway,NOx,1994,204.411,kt,National total for the entire territory (based...
...,...,...,...,...,...,...
68,Sweden,NOx,2020,114.813,kt,National total for the entire territory (based...
69,Sweden,NOx,2021,113.085,kt,National total for the entire territory (based...
70,Sweden,NOx,2022,109.063,kt,National total for the entire territory (based...
71,Sweden,NOx,2023,104.184,kt,National total for the entire territory (based...
